# NeuroAtlas — A RAG-Powered Knowledge Assistant for Mental, Neurodevelopmental, Neurological & Sleep Disorders
Domain: mental disorders, neurodevelopmental disorders, neurological/movement disorders, and
sleep disorders, sourced from WHO, NIMH, NINDS, NHLBI, NICHD, NIGMS, CDC and AASM 


## 2.1 Load & Inspect




In [4]:
import pymupdf
def pdf_to_markdown(pdf_path, markdown_path):

    markdown = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    # Creates/opens the output Markdown file in write mode using UTF-8 encoding.
    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown)) #one string

    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown(
    "../data/documents/Neuro_dataset.pdf",
    "../data/documents/output.md"
)



Markdown file created: ../data/documents/output.md


# 2.2 Chunking Strategy

cleaning text


In [91]:
def clean_text(text):

    # Remove page number at the beginning
    text = re.sub(r'^\d+\s*\n', '', text)

    # Fix words broken by a hyphen and spaces
    text = re.sub(r'(\w)-\s+(\w)', r'\1\2', text)

    # Replace tabs/spaces after bullet points with one space
    text = re.sub(r'•\s*', '• ', text)

     # Remove reference/bibliography sections
    text = re.sub(
        r'(?im)^\s*(REFERENCES|BIBLIOGRAPHY)\s*$.*',
        '',
        text,
        flags=re.DOTALL
    )

    # Replace line breaks that occur inside a sentence/paragraph with a space.
    # Keep the line break when the next line starts with a bullet.
    text = re.sub(r'\n(?!•)', ' ', text)

    # Remove repeated PDF footer text
    text = re.sub(r'Copyright © National Academy of Sciences\.', '', text)
    text = re.sub(r'All rights reserved\.', '', text)
    text = re.sub(r'http://www\.nap\.edu/catalog/11617\.html', '', text)

    # Remove extra spaces
    text = re.sub(r'[ \t]+', ' ', text)

    # Clean up spaces around paragraphs
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()

This cell imports the required libraries, defines the page ranges for each source document, provides a function to identify the source book for each page, and loads the extracted Markdown text for the RAG pipeline.

In [92]:
import re
from pathlib import Path
from semantic_chunkers import StatisticalChunker
from semantic_router.encoders import HuggingFaceEncoder

# Book page ranges
BOOK_RANGES = [
    ("WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)", 1, 852),
    ("Fundamentals of Psychological Disorders (mental use)", 853, 1114),
    ("Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental)", 1115, 1622),
    ("Developmental Screening - CDC", 1623, 1641),
    ("Neurological Disorders - Public Health Challenges (neuro, WHO)", 1642, 1873),
    ("Global Status Report on Neurology (neuro 2, WHO)", 1874, 2158),
    ("Sleep Disorders and Sleep Deprivation (National Academies)", 2159, 2583),
    ("Rhythmic Movement Disorder case report (rmd, JCSM)", 2584, 2587),
]

# book names
def get_book_name(page_number):
    """Return the book name corresponding to a PDF page number."""
    for book_name, start_page, end_page in BOOK_RANGES:
        if start_page <= page_number <= end_page:
            return book_name
    return None

MARKDOWN_PATH = Path("../data/documents/output.md")
text = MARKDOWN_PATH.read_text(encoding="utf-8")

print(f"Text length: {len(text):,} characters")


Text length: 7,202,013 characters


This cell splits the Markdown into individual pages, groups consecutive pages by source book, and displays the resulting page and book counts.

In [6]:
def split_into_pages(text):
    """Split Markdown text into individual pages."""
    pattern = r"^##\s*Page\s+(\d+)\s*$"
    parts = re.split(pattern, text, flags=re.MULTILINE)

    pages = []
    for i in range(1, len(parts), 2):
        page_number = int(parts[i])
        page_text = parts[i + 1].strip()
        if page_text:
            pages.append({"page_number": page_number, "text": page_text})
    return pages


def group_pages_by_book(pages):
    """Group consecutive pages that belong to the same book."""
    book_groups = []
    current_book = None
    current_pages = []

    for page in pages:
        book_name = get_book_name(page["page_number"])

        if book_name != current_book:
            if current_pages:
                book_groups.append({"book": current_book, "pages": current_pages})
            current_book = book_name
            current_pages = []

        current_pages.append(page)

    if current_pages:
        book_groups.append({"book": current_book, "pages": current_pages})

    return book_groups


pages = split_into_pages(text)
book_groups = group_pages_by_book(pages)

print(f"Number of pages: {len(pages)}")
print(f"Number of book groups: {len(book_groups)}")
for group in book_groups:
    print(f"{group['book']}: pages {group['pages'][0]['page_number']}-{group['pages'][-1]['page_number']}")


Number of pages: 2581
Number of book groups: 8
WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders): pages 1-852
Fundamentals of Psychological Disorders (mental use): pages 853-1114
Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental): pages 1116-1622
Developmental Screening - CDC: pages 1623-1641
Neurological Disorders - Public Health Challenges (neuro, WHO): pages 1642-1871
Global Status Report on Neurology (neuro 2, WHO): pages 1874-2158
Sleep Disorders and Sleep Deprivation (National Academies): pages 2159-2583
Rhythmic Movement Disorder case report (rmd, JCSM): pages 2584-2587


## Semantic Chunking Setup and Smoke Test

This cell initializes the embedding model and the semantic chunking strategy used to divide the document collection into meaningful sections.

The `HuggingFaceEncoder` uses the `sentence-transformers/all-MiniLM-L6-v2` model to convert text into embeddings. These embeddings allow the `StatisticalChunker` to measure semantic similarity between neighboring parts of the document and identify appropriate chunk boundaries.

The `StatisticalChunker` is configured with a minimum split size of 100 tokens, a maximum split size of 400 tokens, and a window size of 5. The minimum and maximum token limits control the approximate size of the semantic chunks, while the window size determines how neighboring text is considered when detecting semantic changes.

An additional similarity-based merge function, `merge_similar_neighbors()`, is used during experimentation to examine whether adjacent splits are still semantically similar and can safely be combined. Bullet points are protected from automatic merging so that separate list items are not incorrectly joined.

Before applying the strategy to the complete document collection, a smoke test is performed on page 1319. The page is cleaned, passed through the semantic chunker, and its top-level chunks and internal splits are displayed. This allows the chunking behavior to be inspected manually and helps verify that the selected parameters produce meaningful sections before processing the entire dataset.

In [93]:
from semantic_chunkers import ConsecutiveChunker
from semantic_chunkers import CumulativeChunker
import numpy as np


encoder = HuggingFaceEncoder(
    name="sentence-transformers/all-MiniLM-L6-v2"
)

chunker = StatisticalChunker(
    encoder=encoder,
    min_split_tokens=100,  # 100
    max_split_tokens=400,  # 400
    window_size=5
)


# chunker = ConsecutiveChunker(
#     encoder=encoder,
#     score_threshold=0.3
# )

# chunker = CumulativeChunker(
#     encoder=encoder,
#     score_threshold=0.3
# )


# 4. MERGE PASS — re-check adjacent splits, glue back together if still similar
def merge_similar_neighbors(splits, encoder, similarity_threshold=0.6):
    if len(splits) < 2:
        return splits

    embeddings = encoder(
        docs=splits,
        normalize_embeddings=True
    )

    merged = [splits[0]]

    for i in range(1, len(splits)):
        text = splits[i]

        # Don't merge if this starts a new bullet
        if text.startswith("•"):
            merged.append(text)
            continue

        similarity = float(
            np.dot(embeddings[i - 1], embeddings[i])
        )

        if similarity >= similarity_threshold:
            merged[-1] = merged[-1] + " " + text
        else:
            merged.append(text)

    return merged


# # Smoke test on page 1319
# # 3. Get page 1319
# test_page = next(
#     page for page in pages
#     if page["page_number"] == 1319
# )


# # 4. Clean the page
# clean_page_text = clean_text(test_page["text"])


# # 5. Chunk the cleaned page
# test_chunks = chunker([clean_page_text])


# # 6. Get the Chunk object
# chunk = test_chunks[0][0]


# # 7. Display the results
# print(f"Page: {test_page['page_number']}")
# print(f"Number of top-level chunks: {len(test_chunks[0])}")


# for c_idx, chunk in enumerate(test_chunks[0]):

#     print(
#         f"\n===== Chunk {c_idx + 1} "
#         f"({len(chunk.splits)} splits) ====="
#     )

#     for i, split in enumerate(chunk.splits):
#         print(f"\n--- Split {i + 1} ---")
#         print(split)

#     final_splits = merge_similar_neighbors(
#         chunk.splits,
#         encoder,
#         similarity_threshold=0.6
#     )

#     print(
#         f"\n{len(chunk.splits)} splits "
#         f"-> {len(final_splits)} after merge"
#     )

#     for i, s in enumerate(final_splits):
#         print(f"\n--- Merged split {i + 1} ---")
#         print(s)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3433.20it/s]


 Coordinate-Based Header/Footer and Reference Detection

This cell detects repeated headers and footers using their position on the original PDF pages rather than relying only on repeated extracted text. The detection is performed separately for each source book to reduce false matches between different documents.

The cell also identifies reference and bibliography pages using multiple citation patterns. Numbered questions, ordinary name lists, and clock-time values are excluded from the citation scoring to reduce false positives.

The detected running headers and footers are stored in `boilerplate`, while reference pages are marked later by `mark_reference_sections()` before the final RAG chunks are created.

In [136]:
"""
Coordinate-based header/footer removal
=======================================

Detects repeated headers and footers from their actual position
on the PDF page and detects reference/bibliography sections
using citation signals.

Requires the original PDF (Neuro_dataset.pdf).
"""

import re
import pymupdf as fitz
from collections import Counter

# ============================================================
# Reference-page detection
# ============================================================

def is_reference_page(text):
    """Return True if the page contains a References/Bibliography heading."""
    for line in text.splitlines():
        stripped = line.strip()
        stripped = stripped.rstrip(":").rstrip("0123456789")

        if re.fullmatch(r"references", stripped, re.I):
            return True

        if re.fullmatch(r"bibliography", stripped, re.I):
            return True

    return False

# ============================================================
# Header/footer normalization
# ============================================================

def normalize_block(text):
    """Normalize a PDF margin block for comparison."""
    lines = text.splitlines()

    lines = [
        line
        for line in lines
        if not re.fullmatch(r"\s*\d+\s*", line)
    ]

    return re.sub(r"\s+", " ", " ".join(lines)).strip()

# ============================================================
# Running header/footer detection
# ============================================================

def find_running_headers_footers(
    pdf_path,
    book_ranges,
    top_pct=0.12,
    bottom_pct=0.12,
    ratio_threshold=0.20,
    min_abs_count=8,
    min_book_pages_for_ratio=20,
):
    """
    Detect repeated text blocks in the top or bottom margin.
    Detection is performed separately for each book.
    """
    with fitz.open(pdf_path) as doc:
        def book_index(page_number):
            for i, (_, start_page, end_page) in enumerate(book_ranges):
                if start_page <= page_number <= end_page:
                    return i
            return -1

        margin_hits = [Counter() for _ in book_ranges]

        for pdf_page_num in range(1, len(doc) + 1):
            book_idx = book_index(pdf_page_num)

            if book_idx == -1:
                continue

            page = doc[pdf_page_num - 1]
            page_height = page.rect.height
            top_band = page_height * top_pct
            bottom_band = page_height * (1 - bottom_pct)
            seen_this_page = set()

            for block in page.get_text("blocks"):
                x0, y0, x1, y1, text = block[:5]
                normalized = normalize_block(text)

                if not normalized:
                    continue

                if y1 <= top_band or y0 >= bottom_band:
                    seen_this_page.add(normalized)

            for text in seen_this_page:
                margin_hits[book_idx][text] += 1

        boilerplate = set()
        skipped_books = []

        for book_idx, (name, start_page, end_page) in enumerate(book_ranges):
            number_of_pages = end_page - start_page + 1

            if number_of_pages < min_book_pages_for_ratio:
                skipped_books.append((name, number_of_pages))
                continue

            for text, count in margin_hits[book_idx].items():
                ratio = count / number_of_pages

                if ratio >= ratio_threshold and count >= min_abs_count:
                    boilerplate.add(text)

    if skipped_books:
        print("Skipped (too small for ratio detection, check manually):")
        for book in skipped_books:
            print(" -", book)

    return boilerplate

# ============================================================
# Reference citation patterns
# ============================================================

NUMBERED_REF = re.compile(
    r"(?:^|\n)\s*\d{1,3}\.\s"
)

JOURNAL_CITATION = re.compile(
    r"\b\d{1,4}"
    r"(?:\(\w+\.?\s?\d*\))?"
    r"\s*:\s*"
    r"(?!00\b)"
    r"[SsA-Z]?\d{1,5}"
)

APA_AUTHOR = re.compile(
    r"\b[A-Z][a-zA-Z\-]+,\s+[A-Z]\."
)

YEAR_PAREN = re.compile(
    r"\(\d{4}[a-z]?\)"
)

VANCOUVER_AUTHOR = re.compile(
    r"\b[A-Z][a-zA-Z\-]+\s[A-Z]{1,3}\b[,.]"
)

# ============================================================
# Rejoin wrapped lines
# ============================================================

def rejoin_wrapped_lines(text):
    """Join hard-wrapped lines into continuous text."""
    lines = text.splitlines()
    output = []
    buffer = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        buffer.append(stripped)

        if stripped.endswith((".", "!", "?", ":")):
            output.append(" ".join(buffer))
            buffer = []

    if buffer:
        output.append(" ".join(buffer))

    return "\n".join(output)

# ============================================================
# Count numbered reference-style lines
# ============================================================

def count_numbered_refs(joined_text):
    """
    Count numbered reference-style lines while ignoring
    numbered questions.
    """
    count = 0

    for line in joined_text.splitlines():
        stripped = line.strip()

        if not re.match(r"\d{1,3}\.\s", stripped):
            continue

        if stripped.endswith("?"):
            continue

        count += 1

    return count

# ============================================================
# Count author-style citation signals
# ============================================================

def count_author_signals(joined_text):
    """
    Count author patterns only when the same line also contains
    a year or journal-style citation.
    """
    count = 0

    for line in joined_text.splitlines():
        has_author = (
            bool(APA_AUTHOR.search(line))
            or bool(VANCOUVER_AUTHOR.search(line))
        )

        has_year_or_journal = (
            bool(YEAR_PAREN.search(line))
            or bool(JOURNAL_CITATION.search(line))
        )

        if has_author and has_year_or_journal:
            count += 1

    return count

# ============================================================
# Reference citation score
# ============================================================

def reference_signal_score(text):
    """
    Calculate citation-shaped signals per 100 words.
    """
    joined = rejoin_wrapped_lines(text)
    word_count = max(len(joined.split()), 1)
    signals = 0

    signals += count_numbered_refs(joined)
    signals += len(JOURNAL_CITATION.findall(joined))
    signals += len(YEAR_PAREN.findall(joined))
    signals += count_author_signals(joined)

    return signals / (word_count / 100)

# ============================================================
# Mark reference sections
# ============================================================

def mark_reference_sections(
    book_pages,
    score_threshold=4.0,
    continuation_threshold=3.0
):
    """
    Mark reference and bibliography pages within one book.

    A reference section starts when an explicit reference heading
    is found or the score reaches score_threshold.

    Once a reference section has started, continuation pages remain
    marked when their score reaches continuation_threshold.
    """
    in_reference_section = False
    marked = []

    for page in book_pages:
        text = page["text"]
        heading_hit = is_reference_page(text)
        score = reference_signal_score(text)

        # Explicit reference/bibliography heading
        if heading_hit:
            in_reference_section = True
            is_ref = True

        # Continuation of an existing reference section
        elif (
            in_reference_section
            and score >= continuation_threshold
        ):
            is_ref = True

        # Start a new reference section
        elif (
            not in_reference_section
            and score >= score_threshold
        ):
            in_reference_section = True
            is_ref = True

        # Normal prose
        else:
            in_reference_section = False
            is_ref = False

        marked.append({
            **page,
            "is_reference": is_ref,
            "_ref_score": round(score, 1)
        })

    return marked

# ============================================================
# Remove detected boilerplate
# ============================================================

def strip_boilerplate(text, boilerplate):
    """Remove lines matching detected running headers/footers."""
    def normalize_line(line):
        return re.sub(r"\s+", " ", line.strip())

    lines = text.splitlines()

    kept = [
        line
        for line in lines
        if normalize_line(line) not in boilerplate
    ]

    return "\n".join(kept)

# ============================================================
# Detect running headers and footers
# ============================================================

boilerplate = find_running_headers_footers(
    "../data/documents/Neuro_dataset.pdf",
    BOOK_RANGES
)

print(
    f"\nDetected {len(boilerplate)} "
    f"true running headers/footers:"
)

for item in sorted(boilerplate):
    print(" -", item)

Skipped (too small for ratio detection, check manually):
 - ('Developmental Screening - CDC', 19)
 - ('Rhythmic Movement Disorder case report (rmd, JCSM)', 4)

Detected 8 true running headers/footers:
 - Clinical Descriptions and Diagnostic Requirements for ICD-11 Mental, Behavioural or Neurodevelopmental Disorders
 - Copyright © National Academy of Sciences. All rights reserved.
 - DISORDERS WITH BROADER-SPECTRUM EFFECTS
 - Global status report on neurology
 - Neurological disorders: public health challenges
 - SLEEP DISORDERS AND SLEEP DEPRIVATION
 - Sleep Disorders and Sleep Deprivation: An Unmet Public Health Problem http://www.nap.edu/catalog/11617.html
 - neurological disorders: a public health approach


 Create Final RAG Chunks

This cell processes all source books, cleans their pages, applies semantic chunking, preserves page positions, and assigns each final chunk its source book, page range, and token count. The resulting chunks are stored in `all_chunks` for the next processing stages.

In [139]:
### Create Final RAG Chunks

all_chunks = []


def combine_pages_with_offsets(pages):
    """Combine pages while preserving their character positions."""

    combined_text = ""
    page_offsets = []

    for page in pages:

        start = len(combined_text)

        combined_text += page["text"]

        end = len(combined_text)

        page_offsets.append({
            "page_number": page["page_number"],
            "start": start,
            "end": end
        })

        combined_text += "\n\n"

    return combined_text, page_offsets


def get_page_range(start_pos, end_pos, page_offsets):
    """Find the first and last page covered by a text span."""

    page_start = None
    page_end = None

    for page in page_offsets:

        if (
            page_start is None
            and start_pos < page["end"]
        ):
            page_start = page["page_number"]

        if end_pos <= page["end"]:

            page_end = page["page_number"]

            break

    return page_start, page_end


# ============================================================
# Pages removed as reference/bibliography pages
# ============================================================

reference_pages_removed = []


# ============================================================
# Process every book
# ============================================================

for group in book_groups:

    book_name = group["book"]
    book_pages = group["pages"]

    print(
        f"\nProcessing: {book_name}"
    )


    # --------------------------------------------------------
    # Mark reference pages
    # --------------------------------------------------------

    marked_pages = mark_reference_sections(
        book_pages
    )


    # --------------------------------------------------------
    # Clean pages
    # --------------------------------------------------------

    cleaned_pages = []


    for page in marked_pages:

        # Remove reference/bibliography pages
        if page["is_reference"]:

            reference_pages_removed.append(
                page["page_number"]
            )

            continue


        # ----------------------------------------------------
        # Remove detected running headers and footers
        # ----------------------------------------------------

        stripped_text = strip_boilerplate(
            page["text"],
            boilerplate
        )


        # ----------------------------------------------------
        # Apply normal text cleaning
        # ----------------------------------------------------

        cleaned_text = clean_text(
            stripped_text
        )


        # ----------------------------------------------------
        # Skip empty pages
        # ----------------------------------------------------

        if not cleaned_text:
            continue


        # ----------------------------------------------------
        # Skip pages containing only a page number
        # ----------------------------------------------------

        if re.fullmatch(
            r"\d+",
            cleaned_text
        ):
            continue


        cleaned_pages.append({
            "page_number": page["page_number"],
            "text": cleaned_text
        })


    # --------------------------------------------------------
    # Combine pages while preserving positions
    # --------------------------------------------------------

    book_text, page_offsets = combine_pages_with_offsets(
        cleaned_pages
    )


    # --------------------------------------------------------
    # Semantic chunking
    # --------------------------------------------------------

    book_chunks = chunker([
        book_text
    ])


    book_chunk_count = 0
    search_position = 0


    # Each Chunk object represents one semantic chunk
    for chunk in book_chunks[0]:

        # Join the internal splits into the final chunk text
        chunk_text = " ".join(
            chunk.splits
        ).strip()


        # ----------------------------------------------------
        # Find the chunk in the combined book text
        # ----------------------------------------------------

        chunk_start = book_text.find(
            chunk.splits[0],
            search_position
        )


        if chunk_start == -1:

            raise ValueError(
                "Could not locate chunk in source text:\n"
                f"{chunk_text[:200]}"
            )


        chunk_end = (
            chunk_start
            + len(chunk_text)
        )


        # ----------------------------------------------------
        # Determine source page range
        # ----------------------------------------------------

        page_start, page_end = get_page_range(
            chunk_start,
            chunk_end,
            page_offsets
        )


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if (
            page_start in reference_pages_removed
            or page_end in reference_pages_removed
        ):

            raise ValueError(
                f"Reference page "
                f"{page_start}-{page_end} "
                f"unexpectedly reached chunking."
            )


        # ----------------------------------------------------
        # Save final RAG chunk
        # ----------------------------------------------------

        all_chunks.append({
            "text": chunk_text,
            "book": book_name,
            "page_start": page_start,
            "page_end": page_end,
            "token_count": chunk.token_count
        })


        # Continue searching after this chunk
        search_position = chunk_end

        book_chunk_count += 1


    print(
        f"Final chunks from this book: "
        f"{book_chunk_count}"
    )


# ============================================================
# Final checks
# ============================================================

print(
    "\n" + "=" * 60
)

print(
    "FINAL CHUNKING SUMMARY"
)

print(
    "=" * 60
)


print(
    f"Reference pages removed: "
    f"{len(reference_pages_removed):,}"
)


print(
    f"Total final chunks: "
    f"{len(all_chunks):,}"
)


print(
    "\nReference pages removed:"
)


print(
    sorted(reference_pages_removed)
)


Processing: WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)


2026-09-10 07:46:06 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.
100%|██████████| 176/176 [00:19<00:00,  9.10it/s]


Final chunks from this book: 1581

Processing: Fundamentals of Psychological Disorders (mental use)


2026-09-10 07:46:26 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.
100%|██████████| 99/99 [00:08<00:00, 11.49it/s]


Final chunks from this book: 751

Processing: Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental)


...

Final chunks from this book: 444

Processing: Sleep Disorders and Sleep Deprivation (National Academies)


2026-09-10 07:47:03 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.
100%|██████████| 81/81 [00:08<00:00,  9.64it/s]
2026-09-10 07:47:12 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.


Final chunks from this book: 718

Processing: Rhythmic Movement Disorder case report (rmd, JCSM)


100%|██████████| 2/2 [00:00<00:00, 13.29it/s]

Final chunks from this book: 14

FINAL CHUNKING SUMMARY
Reference pages removed: 230
Total final chunks: 5,139

Reference pages removed:
[5, 34, 35, 36, 37, 1129, 1144, 1145, 1164, 1165, 1192, 1193, 1194, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1223, 1224, 1225, 1244, 1245, 1246, 1247, 1248, 1249, 1250, 1251, 1273, 1274, 1275, 1276, 1277, 1278, 1279, 1280, 1281, 1282, 1283, 1303, 1304, 1305, 1306, 1307, 1308, 1325, 1326, 1327, 1328, 1329, 1330, 1331, 1353, 1354, 1355, 1356, 1357, 1358, 1359, 1360, 1361, 1376, 1377, 1378, 1379, 1395, 1396, 1397, 1416, 1417, 1418, 1419, 1420, 1421, 1422, 1423, 1432, 1433, 1434, 1435, 1442, 1443, 1444, 1455, 1456, 1466, 1467, 1468, 1469, 1482, 1483, 1484, 1485, 1486, 1499, 1500, 1501, 1502, 1503, 1515, 1516, 1517, 1518, 1519, 1523, 1524, 1542, 1543, 1544, 1545, 1546, 1568, 1569, 1570, 1571, 1572, 1573, 1574, 1575, 1590, 1591, 1592, 1593, 1594, 1609, 1610, 1611, 1612, 1613, 1658, 1677, 1678, 1692, 1706, 1720, 1721, 1722, 1736, 1737, 1747, 1762, 176

In [138]:
test_pages = [
    829,
    917,
    1008,
    1274,
    1275,
    1721,
    1722,
    2322,
    2346,
    2347,
    2348,
    2349,
    2350,
    2351,
    2529
]

for group in book_groups:

    marked_pages = mark_reference_sections(
        group["pages"]
    )

    for page in marked_pages:

        if page["page_number"] in test_pages:

            print(
                f"Page {page['page_number']} | "
                f"Reference: {page['is_reference']} | "
                f"Score: {page['_ref_score']}"
            )

Page 829 | Reference: False | Score: 0.0
Page 917 | Reference: False | Score: 2.9
Page 1008 | Reference: False | Score: 1.0
Page 1274 | Reference: True | Score: 6.2
Page 1275 | Reference: True | Score: 5.5
Page 1721 | Reference: True | Score: 10.8
Page 1722 | Reference: True | Score: 4.5
Page 2322 | Reference: False | Score: 0.8
Page 2346 | Reference: True | Score: 5.6
Page 2347 | Reference: True | Score: 4.3
Page 2348 | Reference: True | Score: 3.4
Page 2349 | Reference: True | Score: 5.6
Page 2350 | Reference: True | Score: 5.0
Page 2351 | Reference: True | Score: 4.6
Page 2529 | Reference: False | Score: 0.0


In [141]:
bad_pages = [
    1721, 1722,
    1274, 1275,
    2346, 2347, 2348
]

for chunk in all_chunks:

    if chunk["page_start"] in bad_pages:

        print(
            f"Page {chunk['page_start']} still exists in all_chunks:"
        )

        print(chunk["text"][:300])

Safeguard for Oversized Chunks

This cell checks the semantic chunks and recursively splits any chunk larger than 600 tokens. Splits are made near sentence boundaries when possible, and token counts are recalculated for each resulting chunk. The final chunk count and token-size statistics are then displayed to verify the resulting RAG chunks.

In [144]:
import re
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")


def split_large_chunk(chunk, max_tokens=600):
    """Recursively split a chunk until every piece is <= max_tokens."""

    text = chunk["text"]

    token_count = len(enc.encode(text))

    # Already small enough
    if token_count <= max_tokens:
        return [{
            **chunk,
            "text": text,
            "token_count": token_count
        }]

    mid = len(text) // 2

    # Find sentence boundaries
    matches = list(
        re.finditer(r"[.!?]\s+", text)
    )

    if matches:
        split_point = min(
            matches,
            key=lambda m: abs(m.end() - mid)
        ).end()
    else:
        # Fallback to nearest space
        split_point = text.find(" ", mid)

        if split_point == -1:
            split_point = mid

    first_text = text[:split_point].strip()
    second_text = text[split_point:].strip()

    first_chunk = {
        **chunk,
        "text": first_text,
        "token_count": len(enc.encode(first_text))
    }

    second_chunk = {
        **chunk,
        "text": second_text,
        "token_count": len(enc.encode(second_text))
    }

    # Recursively split both pieces if necessary
    return (
        split_large_chunk(first_chunk, max_tokens)
        + split_large_chunk(second_chunk, max_tokens)
    )


final_rag_chunks = []

for chunk in all_chunks:
    final_rag_chunks.extend(
        split_large_chunk(chunk, max_tokens=600)
    )


print(f"Before safeguard: {len(all_chunks):,}")
print(f"After safeguard:  {len(final_rag_chunks):,}")


token_counts = [
    chunk["token_count"]
    for chunk in final_rag_chunks
]
print()
print(f"Total chunks: {len(final_rag_chunks):,}")
print(f"Average tokens: {sum(token_counts) / len(token_counts):.1f}")
print(f"Minimum tokens: {min(token_counts)}")
print(f"Maximum tokens: {max(token_counts)}")

Before safeguard: 5,139
After safeguard:  5,252

Total chunks: 5,252
Average tokens: 240.5
Minimum tokens: 22
Maximum tokens: 587


 Assign IDs and Save Semantic Chunks

This cell assigns a unique `chunk_id` to each final RAG chunk, summarizes the number of chunks generated from each source book, inspects the first five chunks and their metadata, and saves the complete chunk dataset as `semantic_chunks.json` for use in the embedding and vector-store stages.

In [145]:
from collections import Counter
import json
from pathlib import Path

for i, chunk in enumerate(final_rag_chunks):
    chunk["chunk_id"] = i

chunk_counts = Counter(chunk["book"] for chunk in final_rag_chunks)

for book, count in chunk_counts.items():
    print(f"{count:5d} chunks - {book}")

for chunk in final_rag_chunks[:5]:
    print("\n" + "=" * 80)
    print(f"Chunk ID:     {chunk['chunk_id']}")
    print(f"Book:         {chunk['book']}")
    print(f"Pages:        {chunk['page_start']}-{chunk['page_end']}")
    print(f"Token count:  {chunk['token_count']}")
    print(f"Text length:  {len(chunk['text'])}")
    print("\nText:")
    print(chunk["text"][:1000])

OUTPUT_PATH = Path("../data/documents/semantic_chunks.json")

with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(final_rag_chunks, file, ensure_ascii=False, indent=2)

print(f"\nSemantic chunks saved to: {OUTPUT_PATH}")

 1593 chunks - WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)
  760 chunks - Fundamentals of Psychological Disorders (mental use)
 1073 chunks - Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental)
   49 chunks - Developmental Screening - CDC
  593 chunks - Neurological Disorders - Public Health Challenges (neuro, WHO)
  447 chunks - Global Status Report on Neurology (neuro 2, WHO)
  723 chunks - Sleep Disorders and Sleep Deprivation (National Academies)
   14 chunks - Rhythmic Movement Disorder case report (rmd, JCSM)

Chunk ID:     0
Book:         WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)
Pages:        1-4
Token count:  153
Text length:  663

Text:
Clinical descriptions and diagnostic requirements for ICD-11 mental, behavioural and neurodevelopmental disorders Clinical descriptions and diagnostic requirements for ICD-11 mental, behavioural and neurodevelopmental disorders Clinical descriptions and diagnost

## 2.3 Embeddings & Vector Store

In [146]:
import json
from pathlib import Path
import chromadb
import numpy as np


# Load final semantic chunks
INPUT_PATH = Path("../data/documents/semantic_chunks.json")

# open file in read mode
with open(INPUT_PATH, "r", encoding="utf-8") as file:
    final_rag_chunks = json.load(file)


print(f"Loaded {len(final_rag_chunks):,} chunks")


# Create persistent Chroma database
CHROMA_PATH = "../data/vector_store"

#Chroma client whose data is persisted on disk.
client = chromadb.PersistentClient( path=CHROMA_PATH)


COLLECTION_NAME = "neurohealth_chunks"


# Recreate collection when rerunning the notebook
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass


collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "NeuroHealth RAG semantic chunks"}
)


# Generate and store embeddings in batches
BATCH_SIZE = 100

for start in range(0, len(final_rag_chunks), BATCH_SIZE):

    batch = final_rag_chunks[start:start + BATCH_SIZE]

    # Extract only the text from each chunk for embedding
    texts = [
        chunk["text"]
        for chunk in batch
    ]

    # every chunk's text is converted into a numerical vector.
    embeddings = encoder(
        docs=texts,
        normalize_embeddings=True
    )

    #Convert embeddings to normal Python lists
    embeddings = np.asarray(embeddings).tolist()

    #Every Chroma record needs a unique ID.
    ids = [
        str(chunk["chunk_id"])
        for chunk in batch
    ]

    #This stores the source information alongside every embedding.
    metadatas = [
        {
            "book": chunk["book"],
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"]
        }
        for chunk in batch
    ]

    # where the actual storage happens.
    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )

    #SHOW progress
    print(
        f"Stored {min(start + BATCH_SIZE, len(final_rag_chunks)):,}"
        f"/{len(final_rag_chunks):,} chunks"
    )


print("\nVector store created successfully.")
print(f"Collection: {COLLECTION_NAME}")
print(f"Number of stored chunks: {collection.count():,}")
print(f"Persistent path: {CHROMA_PATH}")

Loaded 5,252 chunks
Stored 100/5,252 chunks
Stored 200/5,252 chunks
Stored 300/5,252 chunks
Stored 400/5,252 chunks
Stored 500/5,252 chunks
Stored 600/5,252 chunks
Stored 700/5,252 chunks
Stored 800/5,252 chunks
Stored 900/5,252 chunks
Stored 1,000/5,252 chunks
Stored 1,100/5,252 chunks
Stored 1,200/5,252 chunks
Stored 1,300/5,252 chunks
Stored 1,400/5,252 chunks
Stored 1,500/5,252 chunks
Stored 1,600/5,252 chunks
Stored 1,700/5,252 chunks
Stored 1,800/5,252 chunks
Stored 1,900/5,252 chunks
Stored 2,000/5,252 chunks
Stored 2,100/5,252 chunks
Stored 2,200/5,252 chunks
Stored 2,300/5,252 chunks
Stored 2,400/5,252 chunks
Stored 2,500/5,252 chunks
Stored 2,600/5,252 chunks
Stored 2,700/5,252 chunks
Stored 2,800/5,252 chunks
Stored 2,900/5,252 chunks
Stored 3,000/5,252 chunks
Stored 3,100/5,252 chunks
Stored 3,200/5,252 chunks
Stored 3,300/5,252 chunks
Stored 3,400/5,252 chunks
Stored 3,500/5,252 chunks
Stored 3,600/5,252 chunks
Stored 3,700/5,252 chunks
Stored 3,800/5,252 chunks
Stored 3,9

# Retrieval & Prompting

In [147]:
def retrieve_chunks(query, top_k=5):

    # Generate an embedding for the user's question
    query_embedding = encoder(
        docs=[query],
        normalize_embeddings=True
    )

    # Search Chroma for the most similar chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results

In [148]:
test_questions = [
    "What are the main symptoms of insomnia?",
    "What factors can increase the risk of developing depression?",
    "What are the diagnostic features of autism spectrum disorder?",
    "What are the common symptoms of attention-deficit hyperactivity disorder?",
    "What are the main causes or risk factors for stroke?",
    "What are the common symptoms of epilepsy?",
    "What effects can sleep deprivation have on cognitive performance?",
    "What is rhythmic movement disorder and how does it present?",
    "What are the main challenges in managing neurological disorders?",
    "What screening methods are used to identify developmental problems in children?"
]


for i, question in enumerate(test_questions, start=1):

    results = retrieve_chunks(
        question,
        top_k=5
    )

    print("\n" + "=" * 100)
    print(f"Question {i}: {question}")
    print("=" * 100)

    for j in range(len(results["documents"][0])):

        metadata = results["metadatas"][0][j]

        print(f"\n--- Result {j + 1} ---")
        print(f"Distance: {results['distances'][0][j]:.4f}")
        print(f"Book: {metadata['book']}")
        print(
            f"Pages: "
            f"{metadata['page_start']}-{metadata['page_end']}"
        )
        print(f"Text: {results['documents'][0][j][:700]}")


Question 1: What are the main symptoms of insomnia?

--- Result 1 ---
Distance: 0.5050
Book: Sleep Disorders and Sleep Deprivation (National Academies)
Pages: 2254-2254
Text: INSOMNIA Manifestations and Prevalence Insomnia is the most commonly reported sleep problem (Ohayon, 2002). It is a highly prevalent disorder that often goes unrecognized and untreated despite its adverse impact on health and quality of life (Benca, 2005a) (see also Chapter 4). Insomnia is defined by having difficulty falling asleep, maintaining sleep, or by short sleep duration, despite adequate opportunity for a full night’s sleep. Other insomnia symptoms include daytime consequences, such as tiredness, lack of energy, difficulty concentrating, and/or irritability (Simon and VonKorff, 1997). The diagnostic criteria for primary insomnia include: • Difficulty initiating or maintaining sleep 

--- Result 2 ---
Distance: 0.5840
Book: Fundamentals of Psychological Disorders (mental use)
Pages: 999-999
Text: Insomnia

In [149]:
test_questions_2 = [
    "What are the main diagnostic criteria for generalized anxiety disorder?",
    "What are the core symptoms of major depressive disorder?",
    "What are the main characteristics of intellectual developmental disorder?",
    "What are the early warning signs of developmental delay in children?",
    "What are the main causes and risk factors for dementia?",
    "What are the common symptoms and clinical features of migraine?",
    "How does chronic sleep deprivation affect memory and attention?",
    "What are the main treatments or management approaches for epilepsy?",
    "How is autism spectrum disorder distinguished from other neurodevelopmental disorders?",
    "What are the typical clinical features of sleep apnea?"
]

for i, question in enumerate(test_questions_2, start=1):

    results = retrieve_chunks(
        question,
        top_k=5
    )

    print("\n" + "=" * 100)
    print(f"Question {i}: {question}")
    print("=" * 100)

    for j in range(len(results["documents"][0])):

        print(f"\n--- Result {j + 1} ---")
        print(
            f"Distance: "
            f"{results['distances'][0][j]:.4f}"
        )

        print(
            f"Book: "
            f"{results['metadatas'][0][j]['book']}"
        )

        print(
            f"Pages: "
            f"{results['metadatas'][0][j]['page_start']}-"
            f"{results['metadatas'][0][j]['page_end']}"
        )

        print(
            f"Text: "
            f"{results['documents'][0][j][:1000]}"
        )


Question 1: What are the main diagnostic criteria for generalized anxiety disorder?

--- Result 1 ---
Distance: 0.5466
Book: WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)
Pages: 285-285
Text: Course features • Onset of generalized anxiety disorder may occur at any age. However, the typical age of onset is during the early to mid-30s. • Earlier onset of symptoms is associated with greater impairment of functioning and presence of co-occurring mental disorders. • Severity of generalized anxiety disorder symptoms often fluctuates between threshold and subthreshold forms of the disorder, and full remission of symptoms is uncommon. • Although the clinical features of generalized anxiety disorder generally remain consistent across the lifespan, the content of the individual’s worry may vary over time, and there are differences in worry content among different age groups.

--- Result 2 ---
Distance: 0.5574
Book: WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopme

### RAG Prompting

This section builds the prompt used by the RAG system to generate answers from the retrieved evidence. The model is instructed to answer using only the provided context, avoid unsupported information, and cite the source book and page range for each relevant claim.

In [ ]:
import ollama


def build_rag_prompt(question, results):
    """Build a grounded RAG prompt from retrieved chunks."""

    context_parts = []

    for i in range(len(results["documents"][0])):

        document = results["documents"][0][i]
        metadata = results["metadatas"][0][i]

        source = (
            f"[Source {i + 1}] "
            f"{metadata['book']} "
            f"(pages {metadata['page_start']}-"
            f"{metadata['page_end']})"
        )

        context_parts.append(
            f"{source}\n{document}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a scientific RAG assistant.

Answer the user's question using only the provided context.
- Answer the question directly and naturally, without introductory phrases such as "Based on the provided context" or "According to the context."
- Use only information supported by the context.
- Do not invent or add unsupported facts.
- If the context does not contain enough information, say so.
- Give a clear and concise answer.
- Cite each important claim using the source labels in the form [Source 1], [Source 2], etc.
- Do not use citations that are not provided in the context.
# - Do not include a detail unless it directly answers the question and is clearly supported by the retrieved text.
# - Ignore obvious PDF extraction artifacts, formatting fragments, and incomplete words.
# - Provide a sufficiently detailed answer when the context contains enough relevant information; do not make the answer unnecessarily brief.
# - Include the important supporting details needed to fully answer the question, while avoiding unrelated information and repetition.

User question:
{question}

Retrieved context:
{context}

Answer:
"""

    return prompt




RAG Answer Generation

This cell retrieves the most relevant chunks for a user question, builds a grounded RAG prompt from the retrieved evidence, and sends the prompt to the local `llama3.2` model.

The generated answer and the retrieved evidence are both returned so the answer can be evaluated against its supporting sources.

In [218]:
def generate_rag_answer(question, top_k=5):
    """Retrieve evidence and generate a grounded answer."""

    # Retrieve the most relevant chunks from Chroma
    results = retrieve_chunks(
        question,
        top_k=top_k
    )

    # Build the prompt from the retrieved evidence
    prompt = build_rag_prompt(
        question,
        results
    )

    # Generate the answer using the local LLM
    response = ollama.chat(
        model="llama3.2:latest",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0
        }
    )

    # Extract the generated answer
    answer = response["message"]["content"]

    # Return the answer and retrieved evidence
    return answer, results

In [219]:
while True:
    ## question = "What are the main symptoms of ADHD?"
    question = input("\nEnter your question (q to exit): ")

    if question.lower() == "q":
        print("Exiting...")
        break

    answer, results = generate_rag_answer(
        question,
        top_k=5
    )

    print("=" * 80)
    print("Question:")
    print(question)

    print("\n" + "=" * 80)
    print("Generated Answer:")
    print(answer)

    print("\n" + "=" * 80)
    print("Retrieved Sources:")

    for i in range(len(results["documents"][0])):

        metadata = results["metadatas"][0][i]

        print(
            f"[Source {i + 1}] "
            f"{metadata['book']} | "
            f"Pages {metadata['page_start']}-"
            f"{metadata['page_end']}"
        )

Question:
What symptoms are required for a diagnosis of major depressive disorder?

Generated Answer:
For a diagnosis of major depressive disorder (MDD), an individual must experience a depressed mood for most of the day, for more days than not, for at least two weeks [Source 1]. This feeling of a depressed mood is accompanied by at least five of the following characteristic symptoms occurring for most of the day, nearly every day, during a period lasting at least 2 weeks [Source 4]. The symptoms must include at least one symptom from the affective cluster, such as depressed mood or markedly diminished interest or pleasure in activities [Source 4].

Retrieved Sources:
[Source 1] Fundamentals of Psychological Disorders (mental use) | Pages 930-930
[Source 2] WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders) | Pages 277-277
[Source 3] WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders) | Pages 266-266
[Source 4] WHO ICD-11 CDDR (Mental, Behavioural

KeyboardInterrupt: Interrupted by user

In [220]:
test_questions_4 = [
    "What are the essential features required to diagnose generalized anxiety disorder?",
    "What symptoms are typically present in a major depressive episode?",
    "How can autism spectrum disorder present in young children?",
    "What are the main features of attention-deficit hyperactivity disorder?",
    "What are the signs that a child may have a developmental delay?",
    "What factors are associated with an increased risk of dementia?",
    "What does a typical migraine attack involve?",
    "What treatment options are available for people with epilepsy?",
    "Can rhythmic movement disorder continue beyond childhood?",
    "What are the typical symptoms of obstructive sleep apnea?",
    "How does lack of sleep affect attention and reaction time?",
    "What distinguishes autism spectrum disorder from conditions involving neurological regression?",
    "What are the main causes of disorders of intellectual development?",
    "What are the different ways insomnia can affect a person's daytime functioning?",
    "What are the main risk factors associated with stroke?",
    "How is epilepsy different from a single provoked seizure?",
    "What developmental skills are monitored when assessing a child's development?",
    "What are the main types of headache disorders?",
    "How can sleep disorders affect cognitive performance?",
    "What factors can make epilepsy difficult to manage in some populations?"
]


for i, question in enumerate(test_questions_4, start=1):

    print("\n" + "=" * 100)
    print(f"Question {i}: {question}")
    print("=" * 100)

    answer, results = generate_rag_answer(
        question,
        top_k=5
    )

    print("\nGenerated Answer:")
    print(answer)

    print("\nRetrieved Sources:")

    for j in range(len(results["documents"][0])):

        metadata = results["metadatas"][0][j]

        print(
            f"[Source {j + 1}] "
            f"{metadata['book']} | "
            f"Pages {metadata['page_start']}-"
            f"{metadata['page_end']}"
        )


Question 1: What are the essential features required to diagnose generalized anxiety disorder?

Generated Answer:
Marked symptoms of anxiety are required for diagnosis, manifested in either general apprehensiveness that is not restricted to any particular environmental circumstance (i.e. “free-floating anxiety”) [Source 1], or excessive worry (apprehensive expectation) about negative events occurring in several different aspects of everyday life (e.g. work, finances, health, family) [Source 1]. Anxiety and general apprehensiveness or worry are accompanied by additional characteristic symptoms, such as avoidance, frequent need for reassurance, procrastination, and chronic somatic symptoms [Source 4].

Retrieved Sources:
[Source 1] WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders) | Pages 284-284
[Source 2] WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders) | Pages 277-277
[Source 3] WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Dis

## 2.5 Vision Component

In [ ]:
import kagglehub

# # Download latest version of the CHB-MIT EEG dataset
# path = kagglehub.dataset_download("adibadea/chbmitseizuredataset")

# print("Path to dataset files:", path)

# Download latest version of the dataset
path = kagglehub.dataset_download("meimeizhong/facial-dataset-of-autistic-children")

print("Path to dataset files:", path)

# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mamunhasan2cs/down-syndrome-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
